<h1 align="center">🧠 Knowledge Bases & Propositional Logic</h1>

<p align="center"><i>From storing facts to proving conclusions: knowledge bases, propositional logic and automated inference in Python.</i></p>

<p align="center">
  <img src="https://img.shields.io/badge/Course-Knowledge%20Representation%20%26%20Reasoning-6f42c1" />
  <img src="https://img.shields.io/badge/Python-3.x-blue" />
  <img src="https://img.shields.io/badge/Level-Beginner-brightgreen" />
</p>

<p align="center">
  <a href="https://colab.research.google.com/github/YOUR_USERNAME/YOUR_REPO/blob/main/1-first-order-logic/knowledge_bases_and_propositional_logic.ipynb">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
  </a>
</p>

---

## 📌 At a Glance

| | |
|---|---|
| **Topics** | Knowledge Base · Triples · Truth Tables · Tautology · CNF · Modus Tollens · Entailment |
| **Libraries** | `itertools` (built-in), `sympy` |
| **Run time** | Under 1 minute |

## 🗂️ Contents
1. [Knowledge Base: Facts & Queries](#part-1)
2. [Propositional Logic & Truth Tables](#part-2)
3. [CNF & Inference with SymPy](#part-3)
4. [Summary](#summary)

---

**Setup** (only needed once, e.g. on a fresh Colab runtime):

In [1]:
!pip install -q sympy

<a id="part-1"></a>

## 📚 Part 1 · Knowledge Base: Facts & Queries

> 🎯 **Goal:** Represent knowledge as `(subject, relation, object)` triples, query it, and derive facts that were never stored.

### 1.1 Build the knowledge base

In [2]:
facts = [
    ('socrates', 'is_a',       'human'),
    ('human',    'is_a',       'mortal'),
    ('plato',    'is_a',       'human'),
    ('plato',    'teacher_of', 'aristotle'),
]

print('Knowledge Base contains', len(facts), 'facts:')
for (subject, relation, obj) in facts:
    print(f'   {subject} --{relation}--> {obj}')

Knowledge Base contains 4 facts:
   socrates --is_a--> human
   human --is_a--> mortal
   plato --is_a--> human
   plato --teacher_of--> aristotle


### 1.2 Query function

In [3]:
def query(relation):
    """Return all (subject, object) pairs that have the given relation."""
    return [(s, o) for (s, r, o) in facts if r == relation]

print('is_a facts :', query('is_a'))
print('teacher_of :', query('teacher_of'))

is_a facts : [('socrates', 'human'), ('human', 'mortal'), ('plato', 'human')]
teacher_of : [('plato', 'aristotle')]


### 1.3 Ask specific questions

In [4]:
humans = [s for (s, r, o) in facts if r == 'is_a' and o == 'human']
print('Who is a human?', humans)

socrates_is_a = [o for (s, r, o) in facts if s == 'socrates' and r == 'is_a']
print('Socrates directly is a:', socrates_is_a)

Who is a human? ['socrates', 'plato']
Socrates directly is a: ['human']


### 1.4 Reasoning: derive new facts
Since `socrates is_a human` and `human is_a mortal`, the machine concludes `socrates is_a mortal`, a fact we never stored.

In [5]:
def is_a_chain(entity):
    """Follow is_a links to find everything an entity is (transitive reasoning)."""
    results = []
    known = {entity}
    changed = True
    while changed:
        changed = False
        for (s, r, o) in facts:
            if r == 'is_a' and s in known and o not in known:
                known.add(o)
                results.append(o)
                changed = True
    return results

print('By reasoning, socrates is_a:', is_a_chain('socrates'))

By reasoning, socrates is_a: ['human', 'mortal']


### 1.5 Practice: family knowledge base

In [6]:
family_facts = [
    ('imran', 'parent_of', 'bilal'),
    ('imran', 'parent_of', 'sara'),
    ('hina',  'parent_of', 'bilal'),
    ('bilal', 'parent_of', 'ahmed'),
    ('sara',  'parent_of', 'zoya'),
]

def query_in(fact_list, relation):
    return [(s, o) for (s, r, o) in fact_list if r == relation]

for parent, child in query_in(family_facts, 'parent_of'):
    print(f'{parent} is a parent of {child}')

imran is a parent of bilal
imran is a parent of sara
hina is a parent of bilal
bilal is a parent of ahmed
sara is a parent of zoya


> 💡 **Key Takeaway:** A **fact** is knowledge we *store*; a **query** is a question we ask to *search* it. Reasoning goes one step further and *derives* facts that were never stored.

---
<a id="part-2"></a>

## 🔢 Part 2 · Propositional Logic & Truth Tables

> 🎯 **Goal:** Build a truth-table generator and test whether a formula is satisfiable or a tautology.

### 2.1 Truth table generator

In [7]:
import itertools

def analyse(varnames, formula):
    print(' '.join(varnames), '| F')
    results = []
    # Generates all 2^n True/False assignments
    for values in itertools.product([False, True], repeat=len(varnames)):
        env = dict(zip(varnames, values))
        r = formula(env)
        results.append(r)
        print(' '.join(str(int(env[v])) for v in varnames), '|', int(r))
    print('Satisfiable:', any(results))
    print('Valid (tautology):', all(results))

### 2.2 Example A: $(P \land Q) \lor \neg P$

| Step | Law applied | Expression |
|---|---|---|
| 1 | Original | $(P \land Q) \lor \neg P$ |
| 2 | Reorder | $\neg P \lor (P \land Q)$ |
| 3 | Distributive | $(\neg P \lor P) \land (\neg P \lor Q)$ |
| 4 | Complement ($\neg P \lor P \equiv T$) | $T \land (\neg P \lor Q)$ |
| 5 | Identity ($T \land A \equiv A$) | $\neg P \lor Q$ |

A formula with 2 variables has $2^2 = 4$ rows.

In [8]:
print('(P AND Q) OR (NOT P)')
analyse(['P', 'Q'], lambda e: (e['P'] and e['Q']) or (not e['P']))

(P AND Q) OR (NOT P)
P Q | F
0 0 | 1
0 1 | 1
1 0 | 0
1 1 | 1
Satisfiable: True
Valid (tautology): False


### 2.3 Example B: $P \lor \neg P$

By the **Complement Law (Law of Excluded Middle)**, $P \lor \neg P \equiv T$, so it is a **tautology**.

In [9]:
print('P OR (NOT P)')
analyse(['P'], lambda e: e['P'] or (not e['P']))

P OR (NOT P)
P | F
0 | 1
1 | 1
Satisfiable: True
Valid (tautology): True


### 2.4 Example C: $(P \land Q) \rightarrow R$

| Step | Law applied | Expression |
|---|---|---|
| 1 | Original | $(P \land Q) \rightarrow R$ |
| 2 | Implication ($A \rightarrow B \equiv \neg A \lor B$) | $\neg(P \land Q) \lor R$ |
| 3 | De Morgan | $(\neg P \lor \neg Q) \lor R$ |
| 4 | Associative | $\neg P \lor \neg Q \lor R$ |

In [10]:
print('(P AND Q) -> R')
analyse(['P', 'Q', 'R'], lambda e: not (e['P'] and e['Q']) or e['R'])

(P AND Q) -> R
P Q R | F
0 0 0 | 1
0 0 1 | 1
0 1 0 | 1
0 1 1 | 1
1 0 0 | 1
1 0 1 | 1
1 1 0 | 0
1 1 1 | 1
Satisfiable: True
Valid (tautology): False


> 💡 **Key Takeaway:** **Satisfiable** = true for at least one assignment. **Valid (tautology)** = true for *all* assignments. Only `P ∨ ¬P` above is a tautology.

---
<a id="part-3"></a>

## ⚙️ Part 3 · CNF & Inference with SymPy

> 🎯 **Goal:** Convert formulas to Conjunctive Normal Form and verify inference rules automatically.

### 3.1 Convert `P ↔ Q` to CNF

In [11]:
from sympy import symbols
from sympy.logic.boolalg import to_cnf, Equivalent
from sympy.logic.inference import satisfiable, entails

P, Q, R = symbols('P Q R')

print('CNF of P <-> Q:', to_cnf(Equivalent(P, Q)))

CNF of P <-> Q: (P | ~Q) & (Q | ~P)


$P \leftrightarrow Q \equiv (P \rightarrow Q) \land (Q \rightarrow P) \equiv (\neg P \lor Q) \land (\neg Q \lor P)$

It is a conjunction of clauses, each a disjunction of literals, so it is in **CNF**. The two clauses force $P$ and $Q$ to share the same truth value.

### 3.2 Modus Tollens: $\{P \rightarrow Q,\ \neg Q\} \models \neg P$

In [12]:
kb_modus_tollens = [P >> Q, ~Q]
print('KB entails ~P (Modus Tollens):', entails(~P, kb_modus_tollens))

KB entails ~P (Modus Tollens): True


If $P \rightarrow Q$ holds and $Q$ is false, then $P$ must be false. Resolving $(\neg P \lor Q)$ with $\neg Q$ gives $\neg P$.

### 3.3 Disjunctive Syllogism: $\{P \lor Q,\ \neg P\} \models Q$

In [13]:
kb_syllogism = [P | Q, ~P]
print('KB entails Q (Disjunctive Syllogism):', entails(Q, kb_syllogism))

KB entails Q (Disjunctive Syllogism): True


At least one of $P$, $Q$ is true and $P$ is false, so $Q$ must be true. Resolving $(P \lor Q)$ with $\neg P$ gives $Q$.

> 💡 **Key Takeaway:** Inference rules can be checked mechanically: `entails(query, knowledge_base)` returns `True` when the conclusion follows from the KB.

---
<a id="summary"></a>
## ✅ Summary

| Part | Topic | What you learned |
|---|---|---|
| 1 | Knowledge Base | Triples, queries, transitive reasoning |
| 2 | Propositional Logic | Truth tables, satisfiability, tautology, logical laws |
| 3 | SymPy Inference | CNF conversion, Modus Tollens, Disjunctive Syllogism |

<p align="center"><i>⬅️ Back to the repository README · Next: First-Order Logic ➡️</i></p>